# model_attribution_card_v1

目标：验证 `src/analysis` 的批量模型归因 card，统一输出 feature attribution、top features、capabilities 和中文 review 需要的 summary。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import write_model_attribution_card
from src.data import build_supervised_dataset, load_soxl_support_frame

EXPERIMENT = "model_attribution_card_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)


In [2]:
frame = load_soxl_support_frame(start="2026-01-01", end="2026-04-30")
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= "2026-03-31"].copy()
test = dataset.frame.loc[dataset.frame["date"] > "2026-03-31"].copy()
result = write_model_attribution_card(
    CARD,
    model_names=("ridge_mse", "gcn_mse"),
    train=train,
    test=test,
    feature_columns=dataset.feature_columns[:6],
    top_n=3,
)
summary = dict(result.summary)
summary


{'status': 'completed',
 'schema_version': 'analysis.model_attribution.batch.v1',
 'models': ['ridge_mse', 'gcn_mse'],
 'model_count': 2,
 'train_start': '2026-01-02',
 'train_end': '2026-03-31',
 'test_start': '2026-04-01',
 'test_end': '2026-04-29',
 'feature_count': 6,
 'top_n': 3,
 'attribution_kinds': {'coefficient': 3, 'permutation': 6, 'sensitivity': 6},
 'proxy_models': ['gcn_mse'],
 'artifacts': {'feature_attribution': 'feature_attribution.csv',
  'model_attribution_summaries': 'model_attribution_summaries.json',
  'model_capabilities': 'model_capabilities.csv',
  'top_features_by_model': 'top_features_by_model.csv'},
 'risk_notes': ['Permutation 和 sensitivity 是局部解释工具，不等价于因果解释。',
  'proxy_feature_adapter 模型的归因解释的是当前 proxy feature path，不等价于完整论文结构。',
  '短测试窗归因只适合 review，应继续在 walk-forward 和 regime 分段中复验。']}

In [3]:
top_features = result.top_features_by_model.to_dict(orient="records")
notebook_summary = {
    "schema_version": "research.model_attribution_card_v1",
    "status": "completed",
    "models": summary["models"],
    "model_count": summary["model_count"],
    "feature_count": summary["feature_count"],
    "top_n": summary["top_n"],
    "proxy_models": summary["proxy_models"],
    "attribution_kinds": summary["attribution_kinds"],
    "top_features_by_model_rows": len(result.top_features_by_model),
    "feature_attribution_rows": len(result.feature_attribution),
    "artifacts": summary["artifacts"],
    "top_features_sample": top_features[:6],
}
(CARD / "notebook_summary.json").write_text(json.dumps(notebook_summary, ensure_ascii=False, indent=2), encoding="utf-8")
notebook_summary


{'schema_version': 'research.model_attribution_card_v1',
 'status': 'completed',
 'models': ['ridge_mse', 'gcn_mse'],
 'model_count': 2,
 'feature_count': 6,
 'top_n': 3,
 'proxy_models': ['gcn_mse'],
 'attribution_kinds': {'coefficient': 3, 'permutation': 6, 'sensitivity': 6},
 'top_features_by_model_rows': 6,
 'feature_attribution_rows': 15,
 'artifacts': {'feature_attribution': 'feature_attribution.csv',
  'model_attribution_summaries': 'model_attribution_summaries.json',
  'model_capabilities': 'model_capabilities.csv',
  'top_features_by_model': 'top_features_by_model.csv'},
 'top_features_sample': [{'model': 'gcn_mse',
   'feature': 'soxx_return_5d',
   'normalized_importance': 1.0717671327226306,
   'rank': 1},
  {'model': 'gcn_mse',
   'feature': 'soxl_soxx_relative_5d',
   'normalized_importance': 0.5544540634290123,
   'rank': 2},
  {'model': 'gcn_mse',
   'feature': 'soxx_return_1d',
   'normalized_importance': 0.1993979319824721,
   'rank': 3},
  {'model': 'ridge_mse',
   '

In [4]:
assert summary["schema_version"] == "analysis.model_attribution.batch.v1"
assert summary["models"] == ["ridge_mse", "gcn_mse"]
assert summary["proxy_models"] == ["gcn_mse"]
assert summary["attribution_kinds"]["sensitivity"] >= 2
assert result.top_features_by_model.groupby("model").size().max() == 3
assert (CARD / "feature_attribution.csv").exists()
assert (CARD / "top_features_by_model.csv").exists()
assert (CARD / "model_capabilities.csv").exists()
assert (CARD / "model_attribution_summaries.json").exists()


## 中文结论

- 做了什么：新增并验证批量 `model_attribution_card`，把多个模型的 feature attribution、top features、capability/fidelity 和模型归因 summary 写成统一 sidecar。
- 核心指标：样例包含 `ridge_mse` 和 `gcn_mse`，使用 6 个特征、top_n=3；输出 `feature_attribution.csv`、`top_features_by_model.csv`、`model_capabilities.csv` 和 `model_attribution_summaries.json`。
- regime 差异：本 notebook 只验证解释层 card，不做 regime 切片；后续应把 attribution rows 和 `src/analysis` 的 regime/window segment 联动。
- 是否晋升：晋升的是 analysis 解释层框架能力，不晋升任何模型或交易策略。
- 风险：permutation/sensitivity 是局部解释，不是因果解释；proxy graph 模型解释的是当前 proxy feature path，不代表完整论文结构。
- 下一步：在 dashboard 页面加入 attribution card 入口，并按 regime/window 输出分段 attribution。